# MAHIR Data Generation: NSE 5-minute OHLCV

**Author:** Mahir Sujalkumar Shah | companion to `MAHIR Simulation.ipynb` and the paper *MAHIR: A Bounded-Impact Market-State Framework for Indian Equity Execution Simulation and Its Limits as a Short-Horizon Forecaster*

This notebook does one job: **download 5-minute bars for 20 NSE stocks, check them, and package them for GitHub.** The simulation notebook then reads the files from a raw GitHub URL, so the study always runs on the same frozen data.

**What you get (in `OHLCVData/` and `OHLCVData.zip`)**
* `RELIANCE_NS.csv`, `TCS_NS.csv`, ... : one file per stock, columns `Datetime,Open,High,Low,Close,Volume`, timestamps in IST (+05:30)
* `manifest.json` : source, library versions, download time, requested window, and for every file the rows, sessions, first/last date, missing-bar share, zero-volume bars and a SHA-256 hash
* `README.md` : a full data card (contents, schema, provenance, cleaning rules, per-stock quality table, loading and integrity-check snippets)

**Steps:** run all cells, download `OHLCVData.zip`, unzip it, upload the folder to GitHub, open any CSV on GitHub, click **Raw**, and copy the URL up to the folder name. Paste that folder URL into `GITHUB_BASE_URL` in the simulation notebook.

**Before you make the repository public:** the data come from a third-party vendor (`tvDatafeed` is an unofficial TradingView client). Vendor and exchange terms may restrict redistribution. Check them first; a private repository, or publishing only the code and derived tables, is the safe choice. Whatever you publish, the paper must name the real source and download date.


In [1]:
# =====================================================================================
# CELL 1 — INSTALL & IMPORT
# tvDatafeed is an unofficial TradingView client installed from GitHub.
# =====================================================================================
!pip -q install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git

import os, io, json, time, hashlib, zipfile, shutil, platform, datetime as dt
import numpy as np, pandas as pd
try:
    from tvDatafeed import TvDatafeed, Interval
    TV_OK = True
except Exception as _e:
    TV_OK = False; print("tvDatafeed not available:", _e)
print("Python", platform.python_version(), "| pandas", pd.__version__, "| tvDatafeed available:", TV_OK)


  Preparing metadata (setup.py) ... done
Python 3.13.16 | pandas 2.2.3 | tvDatafeed available: True


In [2]:
# =====================================================================================
# CELL 2 — CONFIGURATION (edit here only)
# SOURCE  : 'tradingview' (tvDatafeed, up to N_BARS bars ~ 66 sessions)
# START/END: requested window; END is exclusive (last bar = 9 Oct 2026). Bars outside it are dropped.
# =====================================================================================
CONFIG = dict(
    TICKERS = ["RELIANCE","TCS","HDFCBANK","INFY","ICICIBANK","SBIN","ITC","LT","BHARTIARTL","HINDUNILVR",
               "AXISBANK","MARUTI","SUNPHARMA","TITAN","NTPC",                       # 15 large-cap
               "FEDERALBNK","IDFCFIRSTB","PNB","NHPC","YESBANK"],                    # 5 lower-liquidity
    SUFFIX = ".NS", EXCHANGE = "NSE",
    SOURCE = "tradingview",
    N_BARS = 5000,                        # tvDatafeed maximum
    START = "2026-07-16", END = "2026-10-10",
    MIN_SESSIONS = 60,                    # warn if a stock has fewer sessions than the paper's 60
    OUT_DIR = "OHLCVData",
    TV_USER = None, TV_PASS = None,       # optional TradingView login (anonymous access may be limited)
)
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)
print(f"{len(CONFIG['TICKERS'])} stocks | source = {CONFIG['SOURCE']} | window {CONFIG['START']} -> {CONFIG['END']} | run date {dt.date.today()}")


20 stocks | source = tradingview | window 2026-07-16 -> 2026-10-10 | run date 2026-10-17


In [3]:
# =====================================================================================
# CELL 3 — DOWNLOAD + CLEANING FUNCTIONS
# clean_bars(): IST timestamps, NSE continuous session 09:15-15:30 only, positive prices, no duplicates.
# _ist_fix()  : tvDatafeed returns naive timestamps; we check whether they fit NSE hours as IST or as UTC.
# =====================================================================================
def clean_bars(df):
    df = df.copy(); df["Datetime"] = pd.to_datetime(df["Datetime"])
    if df["Datetime"].dt.tz is None: df["Datetime"] = df["Datetime"].dt.tz_localize("UTC")
    df["Datetime"] = df["Datetime"].dt.tz_convert("Asia/Kolkata")
    t = df["Datetime"].dt.time
    df = df[(t >= dt.time(9, 15)) & (t < dt.time(15, 30))]
    df = df[["Datetime","Open","High","Low","Close","Volume"]].dropna()
    df = df[(df[["Open","High","Low","Close"]] > 0).all(axis=1)]
    return df.drop_duplicates("Datetime").sort_values("Datetime").reset_index(drop=True)

def _ist_fix(idx):
    idx = pd.DatetimeIndex(idx)
    if idx.tz is not None: return idx.tz_convert("Asia/Kolkata")
    def frac(ix): return np.mean([(x >= dt.time(9, 15)) and (x < dt.time(15, 30)) for x in ix.time])
    as_utc = idx.tz_localize("UTC").tz_convert("Asia/Kolkata")
    return idx.tz_localize("Asia/Kolkata") if frac(idx) >= frac(as_utc) else as_utc

_TV = None
def fetch_tv(code, retries=3, pause=3.0):
    global _TV
    if not TV_OK: raise RuntimeError("tvDatafeed is not installed")
    if _TV is None: _TV = TvDatafeed(CONFIG["TV_USER"], CONFIG["TV_PASS"]) if CONFIG["TV_USER"] else TvDatafeed()
    last = None
    for k in range(retries):
        try:
            d = _TV.get_hist(symbol=code, exchange=CONFIG["EXCHANGE"], interval=Interval.in_5_minute, n_bars=CONFIG["N_BARS"])
            if d is not None and len(d):
                d = d.reset_index(); d = d.rename(columns={d.columns[0]: "Datetime"})
                d.columns = ["Datetime"] + [c.capitalize() for c in d.columns[1:]]
                d["Datetime"] = _ist_fix(d["Datetime"])
                return clean_bars(d[["Datetime","Open","High","Low","Close","Volume"]])
            last = "empty response"
        except Exception as e: last = str(e)
        time.sleep(pause * (k + 1))
    raise RuntimeError(f"{code}: {last}")

def window(df):
    lo = pd.Timestamp(CONFIG["START"], tz="Asia/Kolkata"); hi = pd.Timestamp(CONFIG["END"], tz="Asia/Kolkata")
    d = df[(df.Datetime >= lo) & (df.Datetime < hi)]
    now = pd.Timestamp.now(tz="Asia/Kolkata")                      # drop a still-forming last bar
    if len(d) and d.Datetime.iloc[-1].date() == now.date() and now.time() < dt.time(15, 35): d = d.iloc[:-1]
    return d.reset_index(drop=True)
print("functions ready")


functions ready


In [4]:
# =====================================================================================
# CELL 4 — DOWNLOAD ALL STOCKS, CHECK THEM, SAVE CSVs
# One CSV per stock: <SYMBOL>_NS.csv. A stock that fails is reported, not silently skipped.
# =====================================================================================
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""): h.update(blk)
    return h.hexdigest()

files, failed = [], []
for code in CONFIG["TICKERS"]:
    sym = code + CONFIG["SUFFIX"]; stem = sym.replace(".", "_"); path = os.path.join(CONFIG["OUT_DIR"], stem + ".csv")
    try:
        raw = fetch_tv(code)
        df = window(raw)
        if len(df) == 0: raise RuntimeError("no bars inside the requested window")
        df.to_csv(path, index=False)
        per = df.groupby(df.Datetime.dt.date).size()
        files.append(dict(file=stem + ".csv", symbol=sym, rows=int(len(df)), sessions=int(len(per)),
                          first=str(df.Datetime.min()), last=str(df.Datetime.max()),
                          bars_per_session_min=int(per.min()), bars_per_session_max=int(per.max()),
                          missing_bar_share_pct=round(100 * (1 - per.mean() / 75), 2),
                          zero_volume_bars=int((df.Volume == 0).sum()),
                          close_min=round(float(df.Close.min()), 2), close_max=round(float(df.Close.max()), 2),
                          median_bar_volume=int(df.Volume.median()), total_volume=int(df.Volume.sum()),
                          tier="large-cap" if code in CONFIG["TICKERS"][:15] else "lower-liquidity",
                          sha256=sha256(path)))
        print(f"ok   {sym:<14} {len(df):>5} bars  {len(per):>3} sessions  {df.Datetime.min():%d %b} -> {df.Datetime.max():%d %b %Y}")
        time.sleep(1.5)
    except Exception as e:
        failed.append((sym, str(e)[:100])); print(f"FAIL {sym:<14} {str(e)[:100]}")

assert files, "No stock could be downloaded: check the source, your internet connection and the date window."
Q = pd.DataFrame(files).drop(columns=["sha256"])
display(Q[["symbol", "tier", "rows", "sessions", "first", "last", "missing_bar_share_pct", "zero_volume_bars"]])
short = Q[Q.sessions < CONFIG["MIN_SESSIONS"]]
if len(short): print(f"\nWARNING: {len(short)} stock(s) have fewer than {CONFIG['MIN_SESSIONS']} sessions (the paper uses 60):", list(short.symbol))
if Q["first"].min()[:10] > CONFIG["START"]: print("WARNING: the source did not reach back to", CONFIG["START"], "- the dataset starts at", Q["first"].min()[:10])
if failed: print("\nFailed downloads:", failed)


ok   RELIANCE.NS     4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   TCS.NS          4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   HDFCBANK.NS     4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   INFY.NS         4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   ICICIBANK.NS    4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   SBIN.NS         4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   ITC.NS          4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   LT.NS           4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   BHARTIARTL.NS   4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   HINDUNILVR.NS   4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   AXISBANK.NS     4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   MARUTI.NS       4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   SUNPHARMA.NS    4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   TITAN.NS        4356 bars   60 sessions  16 Jul -> 09 Oct 2026
ok   NTPC.NS         4356 bars   60 sessions  16

,symbol,tier,rows,sessions,first,last,missing_bar_share_pct,zero_volume_bars
0,RELIANCE.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
1,TCS.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
2,HDFCBANK.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
3,INFY.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
4,ICICIBANK.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
5,SBIN.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
6,ITC.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
7,LT.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
8,BHARTIARTL.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0
9,HINDUNILVR.NS,large-cap,4356,60,2026-07-16 09:15:00+05:30,2026-10-09 15:10:00+05:30,3.2,0


In [5]:
# =====================================================================================
# CELL 5 — MANIFEST, README, ZIP, DOWNLOAD
# The manifest records provenance so the paper can state exactly where the data came from.
# =====================================================================================
import importlib.metadata as md
def _v(p):
    try: return md.version(p)
    except Exception: return "n/a"
SOURCE_NAME = "TradingView via tvDatafeed (unofficial)"
manifest = dict(dataset="MAHIR NSE 5-minute OHLCV", source=SOURCE_NAME, exchange=CONFIG["EXCHANGE"], interval="5 minutes",
                timezone="Asia/Kolkata (IST, +05:30)", columns=["Datetime", "Open", "High", "Low", "Close", "Volume"],
                window_requested=f"{CONFIG['START']} to {CONFIG['END']} (end exclusive)",
                fetched_at_utc=dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
                libraries=dict(pandas=pd.__version__, tvdatafeed=_v("tvdatafeed")),
                cleaning="NSE continuous session 09:15-15:30 IST only; non-positive prices and duplicate timestamps removed; no price adjustment.",
                failed=[dict(symbol=s, reason=r) for s, r in failed], files=files)
json.dump(manifest, open(os.path.join(CONFIG["OUT_DIR"], "manifest.json"), "w"), indent=1)
def _md_table(df):
    cols = list(df.columns); out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows(): out.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(out)
FD = pd.DataFrame(files)
T_cov = FD[["symbol", "tier", "rows", "sessions", "first", "last", "bars_per_session_min", "bars_per_session_max", "missing_bar_share_pct", "zero_volume_bars"]].copy()
T_cov["first"] = T_cov["first"].str[:16]; T_cov["last"] = T_cov["last"].str[:16]
T_cov.columns = ["Symbol", "Tier", "Bars", "Sessions", "First bar (IST)", "Last bar (IST)", "Min bars/session", "Max bars/session", "Missing bars (%)", "Zero-volume bars"]
T_px = FD[["symbol", "close_min", "close_max", "median_bar_volume", "total_volume"]].copy()
T_px.columns = ["Symbol", "Lowest close (INR)", "Highest close (INR)", "Median bar volume", "Total volume"]
T_px["Total volume"] = T_px["Total volume"].map("{:,}".format); T_px["Median bar volume"] = T_px["Median bar volume"].map("{:,}".format)
T_hash = FD[["file", "rows", "sha256"]].copy(); T_hash["sha256"] = T_hash["sha256"].str[:16] + "..."
T_hash.columns = ["File", "Rows", "SHA-256 (first 16 characters; full hash in manifest.json)"]
fail_txt = "None." if not failed else "; ".join(f"{s} ({r})" for s, r in failed)
n_tot, n_sess_min, n_sess_max = int(FD.rows.sum()), int(FD.sessions.min()), int(FD.sessions.max())
readme = f"""# MAHIR NSE 5-minute OHLCV dataset

Frozen 5-minute price and volume bars for {len(files)} NSE-listed stocks, prepared for the MAHIR study
*"MAHIR: A Bounded-Impact Market-State Framework for Indian Equity Execution Simulation and Its Limits as a Short-Horizon Forecaster"*.
`MAHIR_Simulation.ipynb` reads these files, so every run of the simulation uses exactly the same data.

## At a glance

| | |
|---|---|
| Source | {SOURCE_NAME} |
| Exchange | {CONFIG['EXCHANGE']} (cash equities) |
| Bar size | 5 minutes |
| Window requested | {manifest['window_requested']} |
| Coverage obtained | {FD['first'].min()[:10]} to {FD['last'].max()[:10]}, {n_sess_min if n_sess_min == n_sess_max else str(n_sess_min) + ' to ' + str(n_sess_max)} sessions per stock |
| Downloaded (UTC) | {manifest['fetched_at_utc']} |
| Time zone | Asia/Kolkata (IST, +05:30) |
| Stocks | {len(files)}: {int((FD.tier == 'large-cap').sum())} large-cap and {int((FD.tier == 'lower-liquidity').sum())} lower-liquidity |
| Total bars | {n_tot:,} |
| Failed downloads | {fail_txt} |

## Files

* `<SYMBOL>_NS.csv`: one file per stock, for example `RELIANCE_NS.csv`. The dot in the `.NS` symbol becomes an underscore.
* `manifest.json`: machine-readable provenance. For every file it lists rows, sessions, first and last timestamp, bars per session, missing-bar share, zero-volume bars, price range, volume statistics and the full SHA-256 hash.
* `README.md`: this file.

## Schema

| Column | Type | Meaning |
|---|---|---|
| `Datetime` | ISO-8601 timestamp with offset, e.g. `2026-07-20 09:15:00+05:30` | Start time of the 5-minute bar, IST |
| `Open` | float, INR | First price in the bar |
| `High` | float, INR | Highest price in the bar |
| `Low` | float, INR | Lowest price in the bar |
| `Close` | float, INR | Last price in the bar |
| `Volume` | integer | Shares traded in the bar, as reported by the source |

Rows are sorted by `Datetime`. A full NSE session runs from 09:15 to 15:30 IST, which gives 75 bars; the first bar starts at 09:15 and the last at 15:25.

## Collection method

| Setting | Value |
|---|---|
| Client | `tvDatafeed` (installed from https://github.com/rongardF/tvdatafeed.git) |
| Request | exchange `{CONFIG['EXCHANGE']}`, interval 5 minutes, up to {CONFIG['N_BARS']:,} bars per stock |
| Login | {'anonymous' if not CONFIG['TV_USER'] else 'TradingView account'} |
| Retries | 3 per stock, with increasing pauses |
| Library versions | pandas {pd.__version__}, tvdatafeed {_v('tvdatafeed')}, Python {platform.python_version()} |

## Processing

Only these steps were applied after download. Nothing else was changed.

1. Timestamps converted to Asia/Kolkata. The source returns timestamps without a time zone, so the script checks whether they fit NSE trading hours as IST or as UTC and converts accordingly.
2. Only the continuous session was kept: bars starting from 09:15 up to, but not including, 15:30 IST.
3. Rows with missing values or non-positive prices were dropped, and duplicate timestamps were removed.
4. Bars outside the requested window were dropped, and a bar still forming at download time was dropped.
5. No outlier filtering, no gap filling and no additional price adjustment. Whether the vendor already adjusted prices for corporate actions was not verified.

## Coverage and completeness

{_md_table(T_cov)}

"Missing bars" is the average shortfall against 75 bars per session. Large-cap stocks are the first 15 in the list; the last 5 are the lower-liquidity group.

## Price and volume summary

{_md_table(T_px)}

## File inventory

{_md_table(T_hash)}

## Known limitations

* The data come from a retail charting service through an unofficial client, not from an exchange feed. Bars can be missing, volumes can differ from exchange records, and the vendor may revise history. Do not treat the files as an official record.
* Lower-liquidity stocks have more missing and zero-volume bars.
* The source returns a limited number of bars (about 66 sessions of 5-minute data), so the window cannot be extended backwards later.
* The files are a snapshot. Downloading again on another day gives a different window and may give slightly different bars.
* Results computed from this dataset are not directly comparable with results from a different vendor or download date.

## Load the data

```python
import pandas as pd
df = pd.read_csv("RELIANCE_NS.csv", parse_dates=["Datetime"])
print(df.head())
print(df["Datetime"].dt.date.nunique(), "sessions")
```

## Check file integrity

```python
import hashlib, json
m = json.load(open("manifest.json"))
for f in m["files"]:
    h = hashlib.sha256(open(f["file"], "rb").read()).hexdigest()
    print(f["file"], "OK" if h == f["sha256"] else "MODIFIED")
```

## Terms of use

The bars originate from a third-party vendor and from NSE. Vendor and exchange terms may limit redistribution and commercial use, and the vendor library used here is unofficial. Check those terms before making this repository public or reusing the files. The script that produced this dataset is `MAHIR_Data_Generation.ipynb`.

"""
open(os.path.join(CONFIG["OUT_DIR"], "README.md"), "w").write(readme)
ZIP = "OHLCVData.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(CONFIG["OUT_DIR"])): z.write(os.path.join(CONFIG["OUT_DIR"], f), f"OHLCVData/{f}")
print(f"Wrote {len(files)} CSVs + manifest.json + README.md -> {ZIP} ({os.path.getsize(ZIP)/1e6:.1f} MB)")
try:
    from google.colab import files as _cf; _cf.download(ZIP)
except Exception: print("(not in Colab: the zip is in the working directory)")


Wrote 20 CSVs + manifest.json + README.md -> OHLCVData.zip (1.2 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>